# NNsight 0.8 documentation: GPU checks

Literal vLLM blocks from the main Markdown guides. Run all. Each process releases its GPU engine when it exits. Assertions check saved values and generation steps. No API key is required.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl", "torch==2.11.0+cu129", "torchvision==0.26.0+cu129", "torchaudio==2.11.0+cu129", "torchcodec==0.16.0+cu129", "nnsight[vllm]==0.8.0rc1", "--extra-index-url", "https://download.pytorch.org/whl/cu129"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()


In [ ]:
import torch, importlib.metadata as md
print({p: md.version(p) for p in ["nnsight", "vllm", "transformers", "torch"]})
assert torch.cuda.is_available(), "Select a GPU runtime before Run all"
print(torch.cuda.get_device_name())

## llms.md — blocks 59, 60, 61, 62, 63, 65

In [ ]:
from pathlib import Path
import textwrap
script = 'import torch, nnsight\nfrom nnsight.modeling.vllm import VLLM\n\n# llms.md:59\nfrom nnsight.modeling.vllm import VLLM\n\nmodel = VLLM(\n    "openai-community/gpt2", tensor_parallel_size=1, dispatch=True,\n    dtype="float16", gpu_memory_utilization=0.3, max_model_len=64,\n)\n\nprint("PASS llms.md:59", flush=True)\n\n# llms.md:60\nwith model.trace("The Eiffel Tower is in", temperature=0.0, max_tokens=1):\n    logits = model.logits.clone().save()\nnext_token = model.tokenizer.decode(logits.argmax(dim=-1))\n\nassert logits.shape[-1] == 50257\nprint("PASS llms.md:60", flush=True)\n\n# llms.md:61\nwith model.trace("Hello", max_tokens=3, min_tokens=3, temperature=0.0) as tracer:\n    logits = nnsight.save([])\n    for step in tracer.iter[:3]:\n        logits.append(model.logits.clone())\n\nassert len(logits)==3\nprint("PASS llms.md:61", flush=True)\n\n# llms.md:62\nwith model.trace("The Eiffel Tower is in", temperature=0.0, max_tokens=1):\n    hidden = model.transformer.h[8].output.clone().save()\n    model.transformer.h[8].output[:] = 0\n    logits = model.logits.clone().save()\n\nassert hidden.shape[-1]==768 and logits.shape[-1]==50257\nprint("PASS llms.md:62", flush=True)\n\n# llms.md:63\nwith model.trace(max_tokens=3, min_tokens=3) as tracer:\n    with tracer.invoke("Hello", temperature=0.8, top_p=0.95):\n        samples = nnsight.save([])\n        for step in tracer.iter[:3]:\n            samples.append(model.samples.item())\n\nassert len(samples)==3\nprint("PASS llms.md:63", flush=True)\n\n# llms.md:65\nwith model.edit() as (tracer, edit):\n    hidden = model.transformer.h[8].output.clone().save()\noutputs = model.generate(["Hello", "The capital of Japan is"], max_tokens=1, temperature=0.0)\ncaptured_hidden = outputs[1].saves["hidden"]\nmodel.clear_edits()\n\nassert captured_hidden.shape[-1]==768\nprint("PASS llms.md:65", flush=True)\n'
path = Path("/content/llms_59.py")
script = "if __name__ == '__main__':\n" + textwrap.indent(script, "    ")
path.write_text(script)
result = subprocess.run([sys.executable, str(path)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()

## llms.md — blocks 64

In [ ]:
from pathlib import Path
import textwrap
script = 'import torch, nnsight\nfrom nnsight.modeling.vllm import VLLM\n\n# llms.md:64\ntapped_model = VLLM(\n    "openai-community/gpt2", dispatch=True,\n    dtype="float16", gpu_memory_utilization=0.3, max_model_len=64,\n    taps=["transformer.h.8.output"],\n)\nwith tapped_model.trace("Hello", max_tokens=1, temperature=0.0):\n    hidden = tapped_model.transformer.h[8].output.clone().save()\n\nassert hidden.shape[-1]==768\nprint("PASS llms.md:64", flush=True)\n'
path = Path("/content/llms_64.py")
script = "if __name__ == '__main__':\n" + textwrap.indent(script, "    ")
path.write_text(script)
result = subprocess.run([sys.executable, str(path)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()

## NNsight.md — blocks 39

In [ ]:
from pathlib import Path
import textwrap
script = 'import torch, nnsight\nfrom nnsight.modeling.vllm import VLLM\n\n# NNsight.md:39\nfrom nnsight.modeling.vllm import VLLM\n\nmodel = VLLM("openai-community/gpt2", dispatch=True, dtype="float16",\n             gpu_memory_utilization=0.3, max_model_len=64)\nwith model.trace("The Eiffel Tower is in", temperature=0.0, max_tokens=1):\n    model.transformer.h[8].output[:] = 0\n    logits = model.logits.clone().save()\n\nassert logits.shape[-1]==50257\nprint("PASS NNsight.md:39", flush=True)\n'
path = Path("/content/NNsight_39.py")
script = "if __name__ == '__main__':\n" + textwrap.indent(script, "    ")
path.write_text(script)
result = subprocess.run([sys.executable, str(path)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()

In [ ]:
print("ALL 8 GPU DOCUMENTATION BLOCKS PASSED")